# Auto-Retrain System - Google Colab Notebook

Este notebook permite ejecutar el sistema de auto-retraining en Google Colab.

## 1. Setup Inicial

In [ ]:
# Instalar dependencias
!pip install optuna xgboost lightgbm scikit-learn pyyaml joblib pandas numpy schedule

In [ ]:
# AUTOMATIC PROJECT DETECTION
import os
from google.colab import drive
drive.mount('/content/drive')

# Find project folder automatically
drive_root = '/content/drive/MyDrive'
possible_names = ['original', 'auto_retrain_system', 'Auto-Retrain', 'auto-retrain']
PROJECT_PATH = None

for name in possible_names:
    test_path = os.path.join(drive_root, name)
    if os.path.exists(test_path):
        PROJECT_PATH = test_path
        break

if PROJECT_PATH is None:
    # List available folders
    available = os.listdir(drive_root)
    print(f"ERROR: No se encontro la carpeta del proyecto.")
    print(f"Carpetas disponibles en MyDrive: {available}")
    print("ACCION: Copie la carpeta 'original' a su Google Drive (MyDrive/)")
    raise FileNotFoundError("Carpeta del proyecto no encontrada")

print(f"Proyecto encontrado: {PROJECT_PATH}")

# Create required directories if they don't exist
for subdir in ['data', 'models', 'logs', 'scripts']:
    full_path = os.path.join(PROJECT_PATH, subdir)
    os.makedirs(full_path, exist_ok=True)
    print(f"Directorio: {subdir}/ - OK")

# Add to Python path
import sys
sys.path.insert(0, PROJECT_PATH)

## 3. Configuracion de Datos

**Nota**: El archivo de datos debe estar en la carpeta `data/` y su nombre debe coincidir con DATA_PATH en config.yaml.

Archivos CSV disponibles se muestran arriba. Para cambiar el archivo:
```python
# Editar config.yaml: cambiar DATA_PATH a 'data/tu_archivo.csv'
```

In [ ]:
# Change to project directory and verify structure
os.chdir(PROJECT_PATH)

# Verify data file exists
data_path = os.path.join(PROJECT_PATH, 'data')
if os.path.exists(data_path):
    csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
    print(f"Archivos CSV en data/: {csv_files}")
else:
    print("Carpeta data/ vacia - Puede generar datos con generate_data.py")

# Verify config
if os.path.exists(os.path.join(PROJECT_PATH, 'config.yaml')):
    print("config.yaml - OK")
    
    # Show current DATA_PATH from config
    import yaml
    with open(os.path.join(PROJECT_PATH, 'config.yaml')) as f:
        cfg = yaml.safe_load(f)
        print(f'DATA_PATH configurado: {cfg.get("DATA_PATH")}')

from main import AutoRetrainSystem

## 3. Ejecutar un Ciclo de Entrenamiento

In [ ]:
# Inicializar sistema
system = AutoRetrainSystem(os.path.join(PROJECT_PATH, 'config.yaml'))

# Cargar datos de ejemplo o crear datos de prueba
import pandas as pd
import numpy as np

# Generar datos de ejemplo
np.random.seed(42)
n_samples = 1000
X = pd.DataFrame({
    'feature1': np.random.randn(n_samples),
    'feature2': np.random.randn(n_samples),
    'feature3': np.random.randn(n_samples),
})
y = 2*X['feature1'] + 0.5*X['feature2'] - X['feature3'] + np.random.randn(n_samples)*0.1

print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} características")

In [ ]:
# Ejecutar ciclo de entrenamiento
results = system.run_cycle(X, y)

import json
print(json.dumps(results, indent=2))

## 4. Modo Programado (Scheduler)

In [ ]:
# Para ejecutar de forma programada, usar:
# 1. Runtime → Schedule fraction of code
# 2. O usar este código para ejecutar cada N horas:

# import time
# while True:
#     results = system.run_cycle()
#     print(f"Último resultado: {results}")
#     time.sleep(24 * 3600)  # 24 horas

## 5. Ver Modelos Guardados

In [ ]:
# Ver modelos guardados
from src.deployer import ModelDeployer

deployer = ModelDeployer(os.path.join(PROJECT_PATH, 'models'))
versions = deployer.list_versions()

print(f"Modelos guardados: {len(versions)}")
for v in versions[:5]:
    print(f"  - {v['version']}: {v['metrics']}")

## 6. Métricas Históricas

In [ ]:
import matplotlib.pyplot as plt

# Cargar historial de métricas
metrics_history = deployer.get_metrics_history()

if metrics_history:
    mse_values = [m['metrics'].get('mse', 0) for m in metrics_history]
    timestamps = [m['created_at'] for m in metrics_history]
    
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(mse_values)), mse_values, marker='o')
    plt.xlabel('Versión')
    plt.ylabel('MSE')
    plt.title('Evolución del MSE')
    plt.grid(True)
    plt.show()
else:
    print("No hay historial de métricas aún")

---

## Próximos Pasos

1. Subir tus datos a Google Drive
2. Actualizar `config.yaml` con las rutas correctas
3. Configurar el scheduler para ejecución automática
4. Monitorear las métricas en el Dashboard